# Lección 07 — Planificación con Claude

En este notebook vas a construir un sistema de planificación completo:
1. Un **agente planificador** que divide tareas complejas en pasos estructurados (JSON)
2. Un **agente ejecutor** que lleva a cabo cada paso con herramientas
3. Un sistema que combina ambos de punta a punta

In [ ]:
%pip install anthropic python-dotenv -q

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
print("Setup listo.")

## Parte 1 — El Agente Planificador

El planificador recibe un objetivo en lenguaje natural y devuelve un plan estructurado en JSON.
No ejecuta nada — solo planifica. Su output es el input del ejecutor.

In [ ]:
SYSTEM_PLANIFICADOR = """
Sos un agente planificador experto. Tu único trabajo es descomponer objetivos complejos en subtareas.

Cuando recibís un objetivo, generás un plan en JSON con esta estructura EXACTA:
{
  "objetivo": "descripción del objetivo principal",
  "duracion_estimada": "tiempo estimado total",
  "presupuesto_usd": número o null,
  "subtareas": [
    {
      "id": 1,
      "descripcion": "qué hacer",
      "agente": "agente_vuelos | agente_hotel | agente_actividades | agente_presupuesto",
      "prioridad": "alta | media | baja",
      "dependencias": [lista de ids de subtareas que deben completarse antes]
    }
  ]
}

Agentes disponibles:
- agente_vuelos: busca y cotiza vuelos
- agente_hotel: busca y reserva hoteles
- agente_actividades: planifica actividades y tours
- agente_presupuesto: calcula y distribuye presupuesto

Respondé SOLO con el JSON, sin texto adicional.
"""

def planificar(objetivo: str) -> dict:
    """Genera un plan estructurado para un objetivo dado."""
    respuesta = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1500,
        system=SYSTEM_PLANIFICADOR,
        messages=[{"role": "user", "content": objetivo}]
    )
    texto = respuesta.content[0].text.strip()
    # Limpiar posibles backticks de markdown
    if texto.startswith("```"):
        texto = texto.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(texto)


# Generar un plan
objetivo = "Organizá un viaje familiar de 7 días a París para 4 personas con presupuesto de USD 6000"
plan = planificar(objetivo)

print(f"Objetivo: {plan['objetivo']}")
print(f"Duración estimada: {plan['duracion_estimada']}")
print(f"Presupuesto: USD {plan.get('presupuesto_usd', 'N/A')}")
print(f"\nSubtareas ({len(plan['subtareas'])}):")
for t in plan['subtareas']:
    deps = f" (depende de: {t['dependencias']})" if t['dependencias'] else ""
    print(f"  [{t['prioridad'].upper()}] {t['id']}. {t['descripcion']} → {t['agente']}{deps}")

## Parte 2 — El Agente Ejecutor con Herramientas

El ejecutor toma el plan y ejecuta cada subtarea usando herramientas especializadas.
Respeta el orden de dependencias: no ejecuta una tarea hasta que sus dependencias estén completas.

In [ ]:
# Herramientas especializadas
def herramienta_vuelos(descripcion: str, presupuesto_usd: int = None) -> dict:
    """Busca y cotiza vuelos según la descripción."""
    return {
        "resultado": "vuelos_encontrados",
        "detalles": f"Vuelo Buenos Aires → París: USD 1200/persona x4 = USD 4800. Salida: 2026-07-01, regreso: 2026-07-08. Aerolínea: Air France, sin escalas.",
        "costo_usd": 4800
    }

def herramienta_hotel(descripcion: str, presupuesto_usd: int = None) -> dict:
    """Busca y cotiza hoteles."""
    return {
        "resultado": "hotel_encontrado",
        "detalles": "Hotel Familia París 3★ en el 5ème arrondissement. USD 180/noche x7 noches = USD 1260. Incluye desayuno para 4 personas.",
        "costo_usd": 1260
    }

def herramienta_actividades(descripcion: str, presupuesto_usd: int = None) -> dict:
    """Planifica actividades y tours."""
    return {
        "resultado": "actividades_planificadas",
        "detalles": "Día 1: Torre Eiffel (USD 80). Día 2: Louvre (USD 60). Día 3: Versalles (USD 120). Día 4-5: Barrios locales (libre). Día 6: Crucero por el Sena (USD 80). Total actividades: USD 340.",
        "costo_usd": 340
    }

def herramienta_presupuesto(descripcion: str, presupuesto_usd: int = None) -> dict:
    """Calcula y distribuye el presupuesto total."""
    return {
        "resultado": "presupuesto_calculado",
        "detalles": "Distribución: Vuelos USD 4800 (72%) | Hotel USD 1260 (19%) | Actividades USD 340 (5%) | Comidas y extras USD 500 (7%). TOTAL: USD 6900. Excede el presupuesto en USD 900. Recomendación: volar en temporada baja o reducir 1 noche de hotel.",
        "costo_usd": 6900
    }

HERRAMIENTAS_DISPONIBLES = {
    "agente_vuelos": herramienta_vuelos,
    "agente_hotel": herramienta_hotel,
    "agente_actividades": herramienta_actividades,
    "agente_presupuesto": herramienta_presupuesto
}

print("Herramientas de ejecución definidas.")

In [ ]:
def ejecutar_plan(plan: dict, verbose: bool = True) -> dict:
    """Ejecuta un plan respetando el orden de dependencias."""
    
    resultados = {}  # id_tarea → resultado
    completadas = set()
    cola = list(plan['subtareas'])  # copia de la lista
    
    print(f"\nEjecutando plan: {plan['objetivo']}")
    print("=" * 60)
    
    intentos = 0
    max_intentos = len(cola) * 3  # evitar bucles infinitos
    
    while cola and intentos < max_intentos:
        intentos += 1
        tarea = cola.pop(0)
        
        # Verificar si las dependencias están completas
        deps_pendientes = [d for d in tarea['dependencias'] if d not in completadas]
        if deps_pendientes:
            # Volver a encolar al final
            cola.append(tarea)
            continue
        
        # Ejecutar la tarea
        if verbose:
            print(f"\n[{tarea['prioridad'].upper()}] Tarea {tarea['id']}: {tarea['descripcion']}")
            print(f"  Agente: {tarea['agente']}")
        
        herramienta = HERRAMIENTAS_DISPONIBLES.get(tarea['agente'])
        if herramienta:
            resultado = herramienta(
                descripcion=tarea['descripcion'],
                presupuesto_usd=plan.get('presupuesto_usd')
            )
            resultados[tarea['id']] = resultado
            completadas.add(tarea['id'])
            if verbose:
                print(f"  ✓ {resultado['detalles']}")
        else:
            if verbose:
                print(f"  ✗ Agente '{tarea['agente']}' no encontrado")
            completadas.add(tarea['id'])  # marcar como completada igual
    
    return resultados


# Ejecutar el plan generado
resultados = ejecutar_plan(plan)

## Parte 3 — Síntesis Final con Claude

Una vez ejecutado el plan, le pasamos todos los resultados a Claude para que genere un resumen amigable para el usuario.

In [ ]:
def sintetizar_resultados(plan: dict, resultados: dict) -> str:
    """Usa Claude para generar un resumen amigable del plan ejecutado."""
    
    contexto = f"""
Objetivo original: {plan['objetivo']}
Presupuesto: USD {plan.get('presupuesto_usd', 'no especificado')}

Resultados de cada subtarea:
"""
    for id_tarea, resultado in resultados.items():
        tarea = next(t for t in plan['subtareas'] if t['id'] == id_tarea)
        contexto += f"\n- {tarea['descripcion']}: {resultado['detalles']}"
    
    respuesta = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=800,
        system="Sos un asistente de viajes. Generá un resumen claro y amigable del plan de viaje con todos los detalles.",
        messages=[{"role": "user", "content": f"Generá el resumen final del viaje:\n{contexto}"}]
    )
    return respuesta.content[0].text


print("\n" + "="*60)
print("RESUMEN FINAL DEL VIAJE")
print("="*60)
resumen = sintetizar_resultados(plan, resultados)
print(resumen)

## Parte 4 — Replanificación Iterativa

¿Qué pasa si el usuario cambia de idea a mitad del proceso? Mostramos cómo replanificar.

In [ ]:
def replanificar(plan_original: dict, cambio: str, resultados_parciales: dict) -> dict:
    """Ajusta el plan existente según un cambio del usuario."""
    
    completadas_info = ""
    if resultados_parciales:
        completadas_info = "\nSubtareas ya completadas:\n" + "\n".join(
            f"- ID {k}: completada" for k in resultados_parciales.keys()
        )
    
    prompt = f"""Plan original:\n{json.dumps(plan_original, indent=2, ensure_ascii=False)}

{completadas_info}

El usuario solicita este cambio: {cambio}

Generá un plan ajustado que:
1. Mantenga las tareas ya completadas (no las repitas)
2. Modifique o agregue subtareas según el cambio pedido
3. Use la misma estructura JSON que el plan original"""
    
    respuesta = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1500,
        system=SYSTEM_PLANIFICADOR,
        messages=[{"role": "user", "content": prompt}]
    )
    texto = respuesta.content[0].text.strip()
    if texto.startswith("```"):
        texto = texto.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(texto)


# El usuario cambia de idea después de ver el presupuesto
cambio_usuario = "El presupuesto es demasiado alto. Reducí a 5 días y buscá opciones más económicas."

print(f"Cambio solicitado: {cambio_usuario}")
print("\nGenerando plan ajustado...")

plan_ajustado = replanificar(plan, cambio_usuario, {1: resultados.get(1)})

print(f"\nPlan ajustado: {plan_ajustado['objetivo']}")
print(f"Nuevas subtareas ({len(plan_ajustado['subtareas'])}):")
for t in plan_ajustado['subtareas']:
    print(f"  {t['id']}. {t['descripcion']} [{t['prioridad']}]")

## Resumen

| Componente | Lo que construiste |
|---|---|
| Agente planificador | Convierte objetivos en planes JSON estructurados |
| Agente ejecutor | Ejecuta subtareas respetando dependencias |
| Síntesis final | Claude convierte resultados técnicos en lenguaje natural |
| Replanificación | Ajuste del plan según feedback del usuario |

Este patrón es la base de sistemas de agentes sofisticados: separar **qué hacer** (planificador) de **cómo hacerlo** (ejecutor) hace el sistema más modular, testeable y fácil de mejorar.

---
En la **Lección 08** vemos cómo hacer que múltiples agentes trabajen en paralelo.